## Materials Project–Based Screening Space Construction

This notebook constructs a screening space for metal nitrides, oxides,
and sulfides using binary parent compounds from Materials Project.

### Workflow
1. Retrieve and standardize binary parent compounds.
2. Select parents with non-theoretical entries and add explicitly required parents.
3. Generate secondary-metal substitution candidates and remove duplicate compositions.
4. Create screening tables at six current densities (50–500 mA g⁻¹).

### Outputs
Parent libraries, unique candidate compositions, current-specific screening
tables, and summary statistics are saved to `new_screening_space/`.

In [1]:
# ============================================================
# 1. Import dependencies
# ============================================================

from pathlib import Path
from getpass import getpass
import json

import numpy as np
import pandas as pd

from mp_api.client import MPRester
from pymatgen.core import Composition

D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Configure the screening space
# ============================================================

# Create the directory for exported tables and summary files.
OUTPUT_DIR = Path("new_screening_space")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define the 35 host metals used to retrieve binary parent compounds.
METALS = [
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Y", "Zr", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag", "Cd", "Hf",
    "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "La", "Ce",
    "Al", "Ga", "In", "Sn", "Pb", "Bi",
]

print("Number of host metals:", len(METALS))

# Map each anion to its corresponding material category.
ANIONS = {
    "N": "Nitride",
    "O": "Oxide",
    "S": "Sulfide",
}

# Define secondary metals for nominal host-metal substitution.
DOPANTS = [
    "Mn", "Co", "In", "Ru", "Pd", "Ag", "Pt", "Cr", "Ni",
    "Fe", "Ti", "V", "Cu", "Zr", "Nb", "Mo", "W", "Ce",
]

# Fractions of the total parent metal content replaced by a dopant.
DOPANT_RATIOS = [
    0.05,
    0.10,
    0.20,
    0.50,
]

# Current densities used to construct model input tables, in mA g^-1.
CURRENT_DENSITIES = [
    50,
    100,
    200,
    300,
    400,
    500,
]

# True: initially select formulas with at least one non-theoretical entry.
# False: select all retrieved parent formulas, including theoretical ones.
# Explicitly required parent formulas are added separately in Section 8.
USE_CORE_PARENTS_ONLY = True

# Include the original undoped parent compositions in the candidate set.
INCLUDE_UNDOPED_PARENTS = True

print("Screening parameters configured.")


Number of host metals: 35
Screening parameters configured.


In [3]:
# ============================================================
# 3. Read the Materials Project API key
# ============================================================

# Use a hidden prompt to avoid displaying the API key in the notebook.
API_KEY = getpass(
    "Enter your Materials Project API key: "
).strip()

if not API_KEY:
    raise ValueError("The API key must not be empty.")

print("API key loaded without displaying it in the notebook.")

Enter your Materials Project API key: ········
API key loaded without displaying it in the notebook.


In [4]:
# ============================================================
# 4. Define formula and metadata utilities
# ============================================================

def format_amount(
    value: float,
    tolerance: float = 1e-8,
) -> str:
    """Format a stoichiometric coefficient for a chemical formula.

    Parameters
    ----------
    value : float
        Stoichiometric coefficient to format.
    tolerance : float, optional
        Absolute tolerance for treating a value as unity or an integer.

    Returns
    -------
    str
        An empty string for unity, an integer string for integer values,
        or a decimal string with trailing zeros removed.

    Examples
    --------
    1.0 -> ""
    2.0 -> "2"
    0.95 -> "0.95"
    1.80 -> "1.8"
    """
    value = float(value)

    if abs(value - 1.0) < tolerance:
        return ""

    nearest_integer = round(value)

    if abs(value - nearest_integer) < tolerance:
        return str(int(nearest_integer))

    return f"{value:.8f}".rstrip("0").rstrip(".")


def make_binary_formula(
    formula: str,
    metal: str,
    anion: str,
) -> tuple[str, float, float]:
    """Reduce a binary formula and place the metal before the anion.

    Parameters
    ----------
    formula : str
        Chemical formula retrieved from the database.
    metal : str
        Expected metal element symbol.
    anion : str
        Expected anion element symbol.

    Returns
    -------
    tuple[str, float, float]
        Standardized formula, metal coefficient, and anion coefficient.

    Raises
    ------
    ValueError
        If the formula does not contain exactly the expected elements.
    """
    comp = Composition(formula).reduced_composition
    amounts = comp.get_el_amt_dict()

    if set(amounts.keys()) != {metal, anion}:
        raise ValueError(
            f"{formula} is not a strictly binary {metal}-{anion} compound."
        )

    metal_amount = float(amounts[metal])
    anion_amount = float(amounts[anion])

    reduced_formula = (
        f"{metal}{format_amount(metal_amount)}"
        f"{anion}{format_amount(anion_amount)}"
    )

    return (
        reduced_formula,
        metal_amount,
        anion_amount,
    )


def get_symmetry_value(
    symmetry,
    field_name: str,
):
    """Read a symmetry field from a dictionary or an attribute-based object.

    Parameters
    ----------
    symmetry : object or None
        Symmetry metadata returned by the database.
    field_name : str
        Name of the requested symmetry field.

    Returns
    -------
    str or None
        String representation of the field, or None if unavailable.
    """
    if symmetry is None:
        return None

    if isinstance(symmetry, dict):
        value = symmetry.get(field_name)
    else:
        value = getattr(
            symmetry,
            field_name,
            None,
        )

    if value is None:
        return None

    return str(value)


def make_candidate_formula(
    host: str,
    host_amount: float,
    dopant: str | None,
    dopant_amount: float,
    anion: str,
    anion_amount: float,
) -> str:
    """Build a candidate formula in host-metal, dopant, and anion order.

    Parameters
    ----------
    host : str
        Host-metal element symbol.
    host_amount : float
        Host-metal stoichiometric coefficient.
    dopant : str or None
        Secondary-metal element symbol, or None for an undoped parent.
    dopant_amount : float
        Secondary-metal stoichiometric coefficient.
    anion : str
        Anion element symbol.
    anion_amount : float
        Anion stoichiometric coefficient.

    Returns
    -------
    str
        Candidate formula with formatted stoichiometric coefficients.
        The dopant is omitted if absent or its amount is negligible.
    """
    formula = (
        f"{host}{format_amount(host_amount)}"
    )

    if (
        dopant is not None
        and dopant_amount > 1e-12
    ):
        formula += (
            f"{dopant}"
            f"{format_amount(dopant_amount)}"
        )

    formula += (
        f"{anion}{format_amount(anion_amount)}"
    )

    return formula


def make_composition_key(
    amounts: dict[str, float],
) -> str:
    """Generate a scale-independent key for composition deduplication.

    Parameters
    ----------
    amounts : dict[str, float]
        Element symbols mapped to stoichiometric coefficients.
        At least one coefficient must be greater than 1e-12.

    Returns
    -------
    str
        Alphabetically ordered elemental fractions rounded to
        10 decimal places.

    Notes
    -----
    Proportional formulas, such as Fe2O3 and Fe4O6, produce the same key.
    Coefficients less than or equal to 1e-12 are excluded.
    """
    positive_amounts = {
        element: float(amount)
        for element, amount in amounts.items()
        if float(amount) > 1e-12
    }

    total = sum(positive_amounts.values())

    normalized = {
        element: round(amount / total, 10)
        for element, amount in positive_amounts.items()
    }

    return "|".join(
        f"{element}:{normalized[element]:.10f}"
        for element in sorted(normalized)
    )


print("Utility functions defined.")

Utility functions defined.


In [5]:
# ============================================================
# 5. Retrieve binary entries from Materials Project
# ============================================================

# Request composition, provenance, stability, and symmetry metadata.
REQUESTED_FIELDS = [
    "material_id",
    "formula_pretty",
    "elements",
    "nelements",
    "theoretical",
    "is_stable",
    "energy_above_hull",
    "formation_energy_per_atom",
    "symmetry",
]

records = []

with MPRester(
    API_KEY,
    use_document_model=False,
    mute_progress_bars=False,
) as mpr:

    # Restrict requested fields to those exposed by the current API.
    available_fields = set(
        mpr.materials.summary.available_fields
    )

    fields = [
        field
        for field in REQUESTED_FIELDS
        if field in available_fields
    ]

    required_fields = {
        "material_id",
        "formula_pretty",
        "elements",
    }

    missing_required = (
        required_fields - set(fields)
    )

    if missing_required:
        raise RuntimeError(
            "Required Materials Project API fields are unavailable: "
            f"{sorted(missing_required)}"
        )

    print("Fields requested in this query:")
    print(fields)

    for anion, category in ANIONS.items():

        chemical_systems = [
            f"{metal}-{anion}"
            for metal in METALS
        ]

        print("\n" + "=" * 70)
        print(
            f"Querying {category}: "
            f"{len(chemical_systems)} binary chemical systems"
        )
        print("=" * 70)

        docs = mpr.materials.summary.search(
            chemsys=chemical_systems,

            # Retain only binary chemical systems.
            nelements=(2, 2),

            # Exclude deprecated database entries.
            deprecated=False,

            # Exclude GNoMe entries.
            include_gnome=False,

            fields=fields,
            all_fields=False,
            chunk_size=1000,
        )

        print(
            f"Number of structure entries returned for {category}: "
            f"{len(docs)}"
        )

        for doc in docs:

            elements_raw = doc.get(
                "elements",
                [],
            )

            elements = {
                str(element)
                for element in elements_raw
            }

            matched_metals = [
                metal
                for metal in METALS
                if metal in elements
            ]

            # Require exactly one metal from the target host-metal list.
            if len(matched_metals) != 1:
                continue

            metal = matched_metals[0]

            # Confirm that the entry contains only the host and the anion.
            if elements != {metal, anion}:
                continue

            original_formula = doc.get(
                "formula_pretty"
            )

            if not original_formula:
                continue

            try:
                (
                    reduced_formula,
                    metal_amount,
                    anion_amount,
                ) = make_binary_formula(
                    formula=original_formula,
                    metal=metal,
                    anion=anion,
                )

            except Exception as error:
                print(
                    "Skipping an entry that could not be standardized:",
                    original_formula,
                    error,
                )
                continue

            symmetry = doc.get(
                "symmetry"
            )

            records.append({
                "metal": metal,
                "anion": anion,
                "category": category,
                "formula_original": original_formula,
                "reduced_formula": reduced_formula,
                "metal_amount": metal_amount,
                "anion_amount": anion_amount,
                "anion_to_metal_ratio": anion_amount / metal_amount,
                "material_id": str(doc.get("material_id")),
                "theoretical": doc.get("theoretical"),
                "is_stable": doc.get("is_stable"),
                "energy_above_hull_eV_atom": doc.get(
                    "energy_above_hull"
                ),
                "formation_energy_eV_atom": doc.get(
                    "formation_energy_per_atom"
                ),
                "crystal_system": get_symmetry_value(
                    symmetry,
                    "crystal_system",
                ),
                "space_group_symbol": get_symmetry_value(
                    symmetry,
                    "symbol",
                ),
                "space_group_number": get_symmetry_value(
                    symmetry,
                    "number",
                ),
            })

if not records:
    raise RuntimeError(
        "No entries were retrieved. "
        "Check the API key, network connection, and query settings."
    )

all_mp_entries = pd.DataFrame(records)

print("\nData retrieval completed.")
print(
    "Total number of Materials Project structure entries:",
    len(all_mp_entries),
)

display(all_mp_entries.head())

Fields requested in this query:
['material_id', 'formula_pretty', 'elements', 'nelements', 'theoretical', 'is_stable', 'energy_above_hull', 'formation_energy_per_atom', 'symmetry']

Querying Nitride: 35 binary chemical systems


D:\code\TempFiles\ipykernel_57060\3382523215.py:70: MPRestWarning: You have specified fields used by `_search` that can be understood by `search`
   nelements
To ensure long term support, please use their `search` equivalents:
   num_elements
Please see the documentation:
    `search`: https://materialsproject.github.io/api/_autosummary/mp_api.client.routes.materials.summary.SummaryRester.html#mp_api.client.routes.materials.summary.SummaryRester.search
   `_search`: https://api.materialsproject.org/redoc#tag/Materials-Summary/operation/search_materials_summary__get
  docs = mpr.materials.summary.search(
Retrieving SummaryDoc documents: 100%|██████████| 556/556 [00:00<?, ?it/s]


Number of structure entries returned for Nitride: 556

Querying Oxide: 35 binary chemical systems


Retrieving SummaryDoc documents: 100%|██████████| 1513/1513 [00:02<00:00, 662.03it/s]


Number of structure entries returned for Oxide: 1513

Querying Sulfide: 35 binary chemical systems


Retrieving SummaryDoc documents: 100%|██████████| 646/646 [00:00<?, ?it/s]

Number of structure entries returned for Sulfide: 646

Data retrieval completed.
Total number of Materials Project structure entries: 2715


,metal,anion,category,formula_original,reduced_formula,metal_amount,anion_amount,anion_to_metal_ratio,material_id,theoretical,is_stable,energy_above_hull_eV_atom,formation_energy_eV_atom,crystal_system,space_group_symbol,space_group_number
0,Ag,N,Nitride,Ag3N,Ag3N,3.0,1.0,0.333333,mp-aaacebod,True,False,0.980540,0.980540,Tetragonal,I4/mmm,139
1,Ag,N,Nitride,AgN,AgN,1.0,1.0,1.000000,mp-aaacewcr,True,False,1.429242,1.429242,Hexagonal,P6_3/mmc,194
2,Ag,N,Nitride,AgN,AgN,1.0,1.0,1.000000,mp-aaacflte,True,False,1.549490,1.549490,Cubic,F-43m,216
3,Ag,N,Nitride,AgN,AgN,1.0,1.0,1.000000,mp-aaacfltf,True,False,1.544426,1.544426,Hexagonal,P-6m2,187
4,Ag,N,Nitride,AgN,AgN,1.0,1.0,1.000000,mp-aaacidod,True,False,1.419704,1.419704,Cubic,Fm-3m,225


In [6]:
# ============================================================
# 6. Standardize and summarize the parent library
# ============================================================

# ------------------------------------------------------------
# 6.1. Create temporary columns for representative-entry ranking
# ------------------------------------------------------------

# Prefer non-theoretical entries; place unknown values last.
all_mp_entries["_rank_theoretical"] = (
    all_mp_entries["theoretical"]
    .map({
        False: 0,
        True: 1,
    })
    .fillna(2)
)

# Prefer stable entries; place unknown values last.
all_mp_entries["_rank_stable"] = (
    all_mp_entries["is_stable"]
    .map({
        True: 0,
        False: 1,
    })
    .fillna(2)
)

# Prefer lower energy above hull; place missing values last.
all_mp_entries["_rank_hull"] = pd.to_numeric(
    all_mp_entries["energy_above_hull_eV_atom"],
    errors="coerce",
).fillna(np.inf)


# ------------------------------------------------------------
# 6.2. Sort entries within each standardized parent formula
# ------------------------------------------------------------

# Ranking priority:
# non-theoretical entry -> stable entry -> lower energy above hull.
# Use the material ID as the final tie-breaker.
all_mp_entries = (
    all_mp_entries
    .sort_values(
        by=[
            "category",
            "metal",
            "reduced_formula",
            "_rank_theoretical",
            "_rank_stable",
            "_rank_hull",
            "material_id",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6.3. Aggregate entries sharing the same reduced parent formula
# ------------------------------------------------------------

group_columns = [
    "metal",
    "anion",
    "category",
    "reduced_formula",
    "metal_amount",
    "anion_amount",
    "anion_to_metal_ratio",
]

parent_summary = (
    all_mp_entries
    .groupby(
        group_columns,
        as_index=False,
    )
    .agg(
        n_mp_entries=(
            "material_id",
            "nunique",
        ),
        has_non_theoretical_entry=(
            "theoretical",
            lambda x: bool(
                (x == False).any()
            ),
        ),
        has_stable_entry=(
            "is_stable",
            lambda x: bool(
                (x == True).any()
            ),
        ),
        minimum_energy_above_hull_eV_atom=(
            "energy_above_hull_eV_atom",
            "min",
        ),
        minimum_formation_energy_eV_atom=(
            "formation_energy_eV_atom",
            "min",
        ),
        all_material_ids=(
            "material_id",
            lambda x: "; ".join(
                sorted(
                    set(
                        str(value)
                        for value in x
                    )
                )
            ),
        ),
    )
)


# ------------------------------------------------------------
# 6.4. Select one representative entry for each parent formula
# ------------------------------------------------------------

# The first entry for each formula follows the ranking defined above.
representatives = (
    all_mp_entries
    .drop_duplicates(
        subset=[
            "metal",
            "anion",
            "category",
            "reduced_formula",
        ],
        keep="first",
    )
    [
        [
            "metal",
            "anion",
            "category",
            "reduced_formula",
            "material_id",
            "theoretical",
            "is_stable",
            "energy_above_hull_eV_atom",
            "formation_energy_eV_atom",
            "crystal_system",
            "space_group_symbol",
            "space_group_number",
        ]
    ]
    .rename(
        columns={
            "material_id":
                "representative_material_id",
            "theoretical":
                "representative_theoretical",
            "is_stable":
                "representative_is_stable",
            "energy_above_hull_eV_atom":
                "representative_energy_above_hull_eV_atom",
            "formation_energy_eV_atom":
                "representative_formation_energy_eV_atom",
            "crystal_system":
                "representative_crystal_system",
            "space_group_symbol":
                "representative_space_group_symbol",
            "space_group_number":
                "representative_space_group_number",
        }
    )
)

# Combine formula-level summaries with representative-entry metadata.
parent_library_all = (
    parent_summary
    .merge(
        representatives,
        on=[
            "metal",
            "anion",
            "category",
            "reduced_formula",
        ],
        how="left",
    )
    .sort_values(
        by=[
            "category",
            "metal",
            "reduced_formula",
        ]
    )
    .reset_index(drop=True)
)

# The core library requires at least one entry marked theoretical=False.
parent_library_core = (
    parent_library_all[
        parent_library_all["has_non_theoretical_entry"]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Total number of unique parent formulas:",
    len(parent_library_all),
)

print(
    "Number of core parent formulas:",
    len(parent_library_core),
)

print("\nParent counts by material category:")

category_counts = (
    parent_library_all
    .groupby(
        "category",
        as_index=False,
    )
    .agg(
        all_parent_formulas=(
            "reduced_formula",
            "nunique",
        ),
        core_parent_formulas=(
            "has_non_theoretical_entry",
            "sum",
        ),
    )
)

display(category_counts)
display(parent_library_core.head(20))

Total number of unique parent formulas: 721
Number of core parent formulas: 364

Parent counts by material category:


,category,all_parent_formulas,core_parent_formulas
0,Nitride,157,64
1,Oxide,362,177
2,Sulfide,202,123


,metal,anion,category,reduced_formula,metal_amount,anion_amount,anion_to_metal_ratio,n_mp_entries,has_non_theoretical_entry,has_stable_entry,...,minimum_formation_energy_eV_atom,all_material_ids,representative_material_id,representative_theoretical,representative_is_stable,representative_energy_above_hull_eV_atom,representative_formation_energy_eV_atom,representative_crystal_system,representative_space_group_symbol,representative_space_group_number
0,Ag,N,Nitride,AgN,1.0,1.0,1.000000,7,True,False,...,1.419704,mp-aaacewcr; mp-aaacflte; mp-aaacfltf; mp-aaac...,mp-aaackcnp,False,False,1.520130,1.520130,Orthorhombic,Cmce,64
1,Ag,N,Nitride,AgN3,1.0,3.0,3.000000,3,True,False,...,0.350426,mp-aaaaadil; mp-aaabgncz; mp-aaafealj,mp-aaaaadil,False,False,0.354192,0.354192,Orthorhombic,Ibam,72
2,Al,N,Nitride,AlN,1.0,1.0,1.000000,4,True,True,...,-1.679672,mp-aaaaaazl; mp-aaaaabze; mp-aaaaacnk; mp-aaaa...,mp-aaaaaazl,False,True,0.000000,-1.679672,Hexagonal,P6_3mc,186
3,Cd,N,Nitride,Cd3N2,3.0,2.0,0.666667,1,True,False,...,0.416104,mp-aaabgldl,mp-aaabgldl,False,False,0.416104,0.416104,Cubic,Ia-3,206
4,Cd,N,Nitride,CdN6,1.0,6.0,6.000000,1,True,False,...,0.134965,mp-aaabglxf,mp-aaabglxf,False,False,0.134965,0.134965,Orthorhombic,Pbca,61
5,Ce,N,Nitride,CeN,1.0,1.0,1.000000,1,True,True,...,-1.746165,mp-aaaaadrx,mp-aaaaadrx,False,True,0.000000,-1.746165,Cubic,Fm-3m,225
6,Co,N,Nitride,Co2N,2.0,1.0,0.500000,1,True,False,...,0.177865,mp-aaaabhml,mp-aaaabhml,False,False,0.177865,0.177865,Orthorhombic,Pnnm,58
7,Co,N,Nitride,CoN,1.0,1.0,1.000000,4,True,False,...,0.102861,mp-aaaaaarg; mp-aaacfkpd; mp-aaacfkss; mp-aaad...,mp-aaaaaarg,False,False,0.102861,0.102861,Cubic,F-43m,216
8,Co,N,Nitride,CoN12,1.0,12.0,12.000000,2,True,False,...,1.244613,mp-aaacpyjz; mp-aaadcjry,mp-aaacpyjz,False,False,1.244613,1.244613,Triclinic,P-1,2
9,Cr,N,Nitride,Cr2N,2.0,1.0,0.500000,1,True,False,...,-0.254791,mp-aaaaamzs,mp-aaaaamzs,False,False,0.244667,-0.254791,Trigonal,P-31m,162


In [7]:
# ============================================================
# 7. Select the initial parent library
# ============================================================

if USE_CORE_PARENTS_ONLY:
    parents_for_screening = (
        parent_library_core.copy()
    )

    print(
        "Using the core parent library: "
        "has_non_theoretical_entry=True"
    )

else:
    parents_for_screening = (
        parent_library_all.copy()
    )

    print(
        "Using the full parent library, "
        "including theoretical parent formulas."
    )

if parents_for_screening.empty:
    raise RuntimeError(
        "The parent library selected for screening is empty."
    )

print(
    "Number of monometallic parents in the initial screening library:",
    len(parents_for_screening),
)

display(
    parents_for_screening[
        [
            "metal",
            "anion",
            "category",
            "reduced_formula",
            "has_non_theoretical_entry",
            "has_stable_entry",
            "minimum_energy_above_hull_eV_atom",
        ]
    ].head(30)
)


Using the core parent library: has_non_theoretical_entry=True
Number of monometallic parents in the initial screening library: 364


,metal,anion,category,reduced_formula,has_non_theoretical_entry,has_stable_entry,minimum_energy_above_hull_eV_atom
0,Ag,N,Nitride,AgN,True,False,1.419704
1,Ag,N,Nitride,AgN3,True,False,0.350426
2,Al,N,Nitride,AlN,True,True,0.000000
3,Cd,N,Nitride,Cd3N2,True,False,0.416104
4,Cd,N,Nitride,CdN6,True,False,0.134965
5,Ce,N,Nitride,CeN,True,True,0.000000
6,Co,N,Nitride,Co2N,True,False,0.177865
7,Co,N,Nitride,CoN,True,False,0.102861
8,Co,N,Nitride,CoN12,True,False,1.244613
9,Cr,N,Nitride,Cr2N,True,False,0.244667


In [8]:
# ============================================================
# 8. Include explicitly required parent formulas
# ============================================================

# These formulas are retrieved from the full parent library.
# They are included even if they do not satisfy the core-library criterion.
REQUIRED_PARENT_FORMULAS = [
    "Mo3N2",
    "Co4N",
]

if USE_CORE_PARENTS_ONLY:

    required_parents = parent_library_all[
        parent_library_all["reduced_formula"].isin(
            REQUIRED_PARENT_FORMULAS
        )
    ].copy()

    found_required = set(
        required_parents["reduced_formula"]
    )

    missing_required = (
        set(REQUIRED_PARENT_FORMULAS)
        - found_required
    )

    if missing_required:
        raise ValueError(
            "Required parents not found: "
            f"{sorted(missing_required)}"
        )

    # Combine the core library with required parents and remove overlap.
    parents_for_screening = (
        pd.concat(
            [
                parent_library_core,
                required_parents,
            ],
            ignore_index=True,
        )
        .drop_duplicates(
            subset=[
                "metal",
                "anion",
                "category",
                "reduced_formula",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )

    print(
        "Using core parents plus required parents."
    )

else:
    parents_for_screening = (
        parent_library_all.copy()
    )

    print(
        "Using all parent formulas."
    )

if parents_for_screening.empty:
    raise RuntimeError(
        "No parent formulas are available for screening."
    )

print(
    "Number of parent formulas used:",
    len(parents_for_screening),
)

display(
    parents_for_screening[
        [
            "metal",
            "anion",
            "category",
            "reduced_formula",
            "has_non_theoretical_entry",
            "has_stable_entry",
            "minimum_energy_above_hull_eV_atom",
        ]
    ].head(30)
)

Using core parents plus required parents.
Number of parent formulas used: 366


,metal,anion,category,reduced_formula,has_non_theoretical_entry,has_stable_entry,minimum_energy_above_hull_eV_atom
0,Ag,N,Nitride,AgN,True,False,1.419704
1,Ag,N,Nitride,AgN3,True,False,0.350426
2,Al,N,Nitride,AlN,True,True,0.000000
3,Cd,N,Nitride,Cd3N2,True,False,0.416104
4,Cd,N,Nitride,CdN6,True,False,0.134965
5,Ce,N,Nitride,CeN,True,True,0.000000
6,Co,N,Nitride,Co2N,True,False,0.177865
7,Co,N,Nitride,CoN,True,False,0.102861
8,Co,N,Nitride,CoN12,True,False,1.244613
9,Cr,N,Nitride,Cr2N,True,False,0.244667


In [9]:
# ============================================================
# 9. Generate and deduplicate candidate compositions
# ============================================================

candidate_records = []

for _, parent in parents_for_screening.iterrows():

    host = parent["metal"]
    anion = parent["anion"]
    parent_formula = parent["reduced_formula"]

    metal_amount = float(
        parent["metal_amount"]
    )

    anion_amount = float(
        parent["anion_amount"]
    )

    # --------------------------------------------------------
    # 9.1. Include the undoped parent composition
    # --------------------------------------------------------

    if INCLUDE_UNDOPED_PARENTS:

        amounts = {
            host: metal_amount,
            anion: anion_amount,
        }

        formula = make_candidate_formula(
            host=host,
            host_amount=metal_amount,
            dopant=None,
            dopant_amount=0.0,
            anion=anion,
            anion_amount=anion_amount,
        )

        candidate_records.append({
            "compound": formula,
            "composition_key": make_composition_key(amounts),
            "candidate_type": "Undoped parent",
            "parent_formula": parent_formula,
            "host_metal": host,
            "dopant": None,
            "dopant_ratio": 0.0,
            "host_amount": metal_amount,
            "dopant_amount": 0.0,
            "anion": anion,
            "anion_amount": anion_amount,
            "anion_to_total_metal_ratio": anion_amount / metal_amount,
            "category": parent["category"],
            "parent_has_non_theoretical_entry": parent[
                "has_non_theoretical_entry"
            ],
            "parent_has_stable_entry": parent[
                "has_stable_entry"
            ],
            "parent_minimum_energy_above_hull_eV_atom": parent[
                "minimum_energy_above_hull_eV_atom"
            ],
            "representative_material_id": parent[
                "representative_material_id"
            ],
        })

    # --------------------------------------------------------
    # 9.2. Generate nominal secondary-metal substitutions
    # --------------------------------------------------------

    for dopant in DOPANTS:

        # Exclude substitution of the host metal by itself.
        if dopant == host:
            continue

        for ratio in DOPANT_RATIOS:

            # Preserve total metal content and the parent anion content.
            new_host_amount = (
                metal_amount * (1.0 - ratio)
            )

            dopant_amount = (
                metal_amount * ratio
            )

            amounts = {
                host: new_host_amount,
                dopant: dopant_amount,
                anion: anion_amount,
            }

            formula = make_candidate_formula(
                host=host,
                host_amount=new_host_amount,
                dopant=dopant,
                dopant_amount=dopant_amount,
                anion=anion,
                anion_amount=anion_amount,
            )

            # Parent metadata records provenance, not candidate stability.
            candidate_records.append({
                "compound": formula,
                "composition_key": make_composition_key(amounts),
                "candidate_type": "Substitutional doping",
                "parent_formula": parent_formula,
                "host_metal": host,
                "dopant": dopant,
                "dopant_ratio": ratio,
                "host_amount": new_host_amount,
                "dopant_amount": dopant_amount,
                "anion": anion,
                "anion_amount": anion_amount,
                "anion_to_total_metal_ratio": (
                    anion_amount / metal_amount
                ),
                "category": parent["category"],
                "parent_has_non_theoretical_entry": parent[
                    "has_non_theoretical_entry"
                ],
                "parent_has_stable_entry": parent[
                    "has_stable_entry"
                ],
                "parent_minimum_energy_above_hull_eV_atom": parent[
                    "minimum_energy_above_hull_eV_atom"
                ],
                "representative_material_id": parent[
                    "representative_material_id"
                ],
            })

screening_compounds = pd.DataFrame(
    candidate_records
)


# ------------------------------------------------------------
# 9.3. Remove duplicate normalized compositions
# ------------------------------------------------------------

before_deduplication = len(
    screening_compounds
)

# Different parents or substitution routes may yield the same composition.
# Retain the first record after sorting by the specified metadata fields.
screening_compounds = (
    screening_compounds
    .sort_values(
        by=[
            "category",
            "parent_formula",
            "candidate_type",
            "dopant",
            "dopant_ratio",
        ],
        na_position="first",
    )
    .drop_duplicates(
        subset=["composition_key"],
        keep="first",
    )
    .reset_index(drop=True)
)

after_deduplication = len(
    screening_compounds
)

print(
    "Number of candidates before deduplication:",
    before_deduplication,
)

print(
    "Number of unique candidates after deduplication:",
    after_deduplication,
)

print("\nCandidate counts by material category and candidate type:")

candidate_category_counts = (
    screening_compounds
    .groupby(
        [
            "category",
            "candidate_type",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "n_candidates",
        }
    )
)

display(candidate_category_counts)
display(screening_compounds.head(30))

Number of candidates before deduplication: 25726
Number of unique candidates after deduplication: 25012

Candidate counts by material category and candidate type:


,category,candidate_type,n_candidates
0,Nitride,Substitutional doping,4449
1,Nitride,Undoped parent,66
2,Oxide,Substitutional doping,11927
3,Oxide,Undoped parent,177
4,Sulfide,Substitutional doping,8270
5,Sulfide,Undoped parent,123


,compound,composition_key,candidate_type,parent_formula,host_metal,dopant,dopant_ratio,host_amount,dopant_amount,anion,anion_amount,anion_to_total_metal_ratio,category,parent_has_non_theoretical_entry,parent_has_stable_entry,parent_minimum_energy_above_hull_eV_atom,representative_material_id
0,Ag0.95Ce0.05N,Ag:0.4750000000|Ce:0.0250000000|N:0.5000000000,Substitutional doping,AgN,Ag,Ce,0.05,0.95,0.05,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
1,Ag0.9Ce0.1N,Ag:0.4500000000|Ce:0.0500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Ce,0.10,0.90,0.10,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
2,Ag0.8Ce0.2N,Ag:0.4000000000|Ce:0.1000000000|N:0.5000000000,Substitutional doping,AgN,Ag,Ce,0.20,0.80,0.20,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
3,Ag0.5Ce0.5N,Ag:0.2500000000|Ce:0.2500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Ce,0.50,0.50,0.50,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
4,Ag0.95Co0.05N,Ag:0.4750000000|Co:0.0250000000|N:0.5000000000,Substitutional doping,AgN,Ag,Co,0.05,0.95,0.05,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
5,Ag0.9Co0.1N,Ag:0.4500000000|Co:0.0500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Co,0.10,0.90,0.10,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
6,Ag0.8Co0.2N,Ag:0.4000000000|Co:0.1000000000|N:0.5000000000,Substitutional doping,AgN,Ag,Co,0.20,0.80,0.20,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
7,Ag0.5Co0.5N,Ag:0.2500000000|Co:0.2500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Co,0.50,0.50,0.50,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
8,Ag0.95Cr0.05N,Ag:0.4750000000|Cr:0.0250000000|N:0.5000000000,Substitutional doping,AgN,Ag,Cr,0.05,0.95,0.05,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
9,Ag0.9Cr0.1N,Ag:0.4500000000|Cr:0.0500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Cr,0.10,0.90,0.10,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp


In [10]:
# ============================================================
# 10. Build a screening table for each current density
# ============================================================

CURRENT_OUTPUT_DIR = (
    OUTPUT_DIR / "screening_space_by_current"
)

CURRENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Retain each table in memory for subsequent model predictions.
screening_spaces_by_current = {}

for current_density in CURRENT_DENSITIES:

    current_df = screening_compounds.copy()

    # Assign the current density in mA g^-1 to every candidate.
    current_df["CurrentDensity"] = current_density

    # Place the current-density input immediately after the formula.
    preferred_columns = [
        "compound",
        "CurrentDensity",
        "composition_key",
        "candidate_type",
        "parent_formula",
        "host_metal",
        "dopant",
        "dopant_ratio",
        "host_amount",
        "dopant_amount",
        "anion",
        "anion_amount",
        "anion_to_total_metal_ratio",
        "category",
        "parent_has_non_theoretical_entry",
        "parent_has_stable_entry",
        "parent_minimum_energy_above_hull_eV_atom",
        "representative_material_id",
    ]

    # Reorder available columns while retaining any additional columns.
    existing_preferred_columns = [
        column
        for column in preferred_columns
        if column in current_df.columns
    ]

    remaining_columns = [
        column
        for column in current_df.columns
        if column not in existing_preferred_columns
    ]

    current_df = current_df[
        existing_preferred_columns
        + remaining_columns
    ]

    screening_spaces_by_current[
        current_density
    ] = current_df

print(
    "Number of unique compounds:",
    len(screening_compounds),
)

print(
    "Number of current densities:",
    len(CURRENT_DENSITIES),
)

print("\nNumber of rows at each current density:")

for current_density, current_df in (
    screening_spaces_by_current.items()
):
    print(
        f"{current_density} mA g^-1: "
        f"{len(current_df)} rows"
    )

# Preview the screening table at 50 mA g^-1.
display(
    screening_spaces_by_current[50].head(20)
)


Number of unique compounds: 25012
Number of current densities: 6

Number of rows at each current density:
50 mA g^-1: 25012 rows
100 mA g^-1: 25012 rows
200 mA g^-1: 25012 rows
300 mA g^-1: 25012 rows
400 mA g^-1: 25012 rows
500 mA g^-1: 25012 rows


,compound,CurrentDensity,composition_key,candidate_type,parent_formula,host_metal,dopant,dopant_ratio,host_amount,dopant_amount,anion,anion_amount,anion_to_total_metal_ratio,category,parent_has_non_theoretical_entry,parent_has_stable_entry,parent_minimum_energy_above_hull_eV_atom,representative_material_id
0,Ag0.95Ce0.05N,50,Ag:0.4750000000|Ce:0.0250000000|N:0.5000000000,Substitutional doping,AgN,Ag,Ce,0.05,0.95,0.05,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
1,Ag0.9Ce0.1N,50,Ag:0.4500000000|Ce:0.0500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Ce,0.10,0.90,0.10,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
2,Ag0.8Ce0.2N,50,Ag:0.4000000000|Ce:0.1000000000|N:0.5000000000,Substitutional doping,AgN,Ag,Ce,0.20,0.80,0.20,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
3,Ag0.5Ce0.5N,50,Ag:0.2500000000|Ce:0.2500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Ce,0.50,0.50,0.50,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
4,Ag0.95Co0.05N,50,Ag:0.4750000000|Co:0.0250000000|N:0.5000000000,Substitutional doping,AgN,Ag,Co,0.05,0.95,0.05,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
5,Ag0.9Co0.1N,50,Ag:0.4500000000|Co:0.0500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Co,0.10,0.90,0.10,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
6,Ag0.8Co0.2N,50,Ag:0.4000000000|Co:0.1000000000|N:0.5000000000,Substitutional doping,AgN,Ag,Co,0.20,0.80,0.20,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
7,Ag0.5Co0.5N,50,Ag:0.2500000000|Co:0.2500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Co,0.50,0.50,0.50,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
8,Ag0.95Cr0.05N,50,Ag:0.4750000000|Cr:0.0250000000|N:0.5000000000,Substitutional doping,AgN,Ag,Cr,0.05,0.95,0.05,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp
9,Ag0.9Cr0.1N,50,Ag:0.4500000000|Cr:0.0500000000|N:0.5000000000,Substitutional doping,AgN,Ag,Cr,0.10,0.90,0.10,N,1.0,1.0,Nitride,True,False,1.419704,mp-aaackcnp


In [11]:
# ============================================================
# 11. Export screening-space data and summaries
# ============================================================

# ------------------------------------------------------------
# 11.1. Remove temporary ranking columns from the exported data
# ------------------------------------------------------------

all_mp_entries_output = (
    all_mp_entries
    .drop(
        columns=[
            "_rank_theoretical",
            "_rank_stable",
            "_rank_hull",
        ],
        errors="ignore",
    )
)


# ------------------------------------------------------------
# 11.2. Export parent libraries and unique candidate compositions
# ------------------------------------------------------------

# Use UTF-8 with a byte-order mark for compatibility with Excel.
all_mp_entries_output.to_csv(
    OUTPUT_DIR / "01_all_MP_binary_entries.csv",
    index=False,
    encoding="utf-8-sig",
)

parent_library_all.to_csv(
    OUTPUT_DIR / "02_all_unique_parent_formulas.csv",
    index=False,
    encoding="utf-8-sig",
)

parent_library_core.to_csv(
    OUTPUT_DIR / "03_core_parent_formulas.csv",
    index=False,
    encoding="utf-8-sig",
)

screening_compounds.to_csv(
    OUTPUT_DIR / "04_unique_screening_compounds.csv",
    index=False,
    encoding="utf-8-sig",
)

category_counts.to_csv(
    OUTPUT_DIR / "05_parent_counts_by_category.csv",
    index=False,
    encoding="utf-8-sig",
)

candidate_category_counts.to_csv(
    OUTPUT_DIR / "06_candidate_counts_by_category.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 11.3. Export a separate CSV file for each current density
# ------------------------------------------------------------

current_file_records = []

for current_density, current_df in (
    screening_spaces_by_current.items()
):

    file_name = (
        f"screening_space_"
        f"{current_density}_mA_g.csv"
    )

    file_path = (
        CURRENT_OUTPUT_DIR / file_name
    )

    current_df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig",
    )

    # Record the row count and location of each exported table.
    current_file_records.append({
        "CurrentDensity_mA_g": current_density,
        "n_compounds": len(current_df),
        "file_name": file_name,
        "file_path": str(file_path.resolve()),
    })

    print(
        f"Exported: {file_name}, "
        f"{len(current_df)} rows"
    )

current_file_summary = pd.DataFrame(
    current_file_records
)

current_file_summary.to_csv(
    OUTPUT_DIR / "07_current_density_file_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 11.4. Export an Excel summary workbook
# ------------------------------------------------------------

# Include parent libraries, unique compositions, and summary tables.
# Full per-current-density tables are stored in separate CSV files.
excel_path = (
    OUTPUT_DIR / "new_screening_space_summary.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
) as writer:

    all_mp_entries_output.to_excel(
        writer,
        sheet_name="All_MP_entries",
        index=False,
    )

    parent_library_all.to_excel(
        writer,
        sheet_name="All_parent_formulas",
        index=False,
    )

    parent_library_core.to_excel(
        writer,
        sheet_name="Core_parent_formulas",
        index=False,
    )

    screening_compounds.to_excel(
        writer,
        sheet_name="Unique_candidates",
        index=False,
    )

    category_counts.to_excel(
        writer,
        sheet_name="Parent_counts",
        index=False,
    )

    candidate_category_counts.to_excel(
        writer,
        sheet_name="Candidate_counts",
        index=False,
    )

    current_file_summary.to_excel(
        writer,
        sheet_name="Current_file_summary",
        index=False,
    )


Exported: screening_space_50_mA_g.csv, 25012 rows
Exported: screening_space_100_mA_g.csv, 25012 rows
Exported: screening_space_200_mA_g.csv, 25012 rows
Exported: screening_space_300_mA_g.csv, 25012 rows
Exported: screening_space_400_mA_g.csv, 25012 rows
Exported: screening_space_500_mA_g.csv, 25012 rows


In [12]:
# ============================================================
# 12. Display the final summary
# ============================================================

print("\n" + "=" * 70)
print("Screening-space generation completed")
print("=" * 70)

print(
    "Total number of unique parent formulas:",
    len(parent_library_all),
)

print(
    "Number of core parent formulas:",
    len(parent_library_core),
)

print(
    "Final number of unique candidate compounds:",
    len(screening_compounds),
)

print(
    "Number of current-density files:",
    len(screening_spaces_by_current),
)

print(
    "\nMain output directory:",
    OUTPUT_DIR.resolve(),
)

print(
    "Current-density output directory:",
    CURRENT_OUTPUT_DIR.resolve(),
)

print(
    "Excel summary workbook:",
    excel_path.resolve(),
)

display(current_file_summary)


Screening-space generation completed
Total number of unique parent formulas: 721
Number of core parent formulas: 364
Final number of unique candidate compounds: 25012
Number of current-density files: 6

Main output directory: D:\code\2\air\new_screening_space
Current-density output directory: D:\code\2\air\new_screening_space\screening_space_by_current
Excel summary workbook: D:\code\2\air\new_screening_space\new_screening_space_summary.xlsx


,CurrentDensity_mA_g,n_compounds,file_name,file_path
0,50,25012,screening_space_50_mA_g.csv,D:\code\2\air\new_screening_space\screening_sp...
1,100,25012,screening_space_100_mA_g.csv,D:\code\2\air\new_screening_space\screening_sp...
2,200,25012,screening_space_200_mA_g.csv,D:\code\2\air\new_screening_space\screening_sp...
3,300,25012,screening_space_300_mA_g.csv,D:\code\2\air\new_screening_space\screening_sp...
4,400,25012,screening_space_400_mA_g.csv,D:\code\2\air\new_screening_space\screening_sp...
5,500,25012,screening_space_500_mA_g.csv,D:\code\2\air\new_screening_space\screening_sp...
